# freeCAM Python-owned FKESSLER Notebook

This is the single maintained freeCAM Notebook. A normal one-process Jupyter kernel controls 24 Python MPI workers through the existing authenticated socket/PBS controller.

There is one model path: `start()` performs pure-Python initialization and every persistent state array is owned by NumPy. The Python `DeviceRegistry` connects StatePool fields to generated thin ABI adapters, and each adapter calls the unchanged original CAM-SIMA Fortran scheme. The old cam_init/cam_run wrapper backend and the hand-maintained Kessler algorithm copy have been removed.

## Runtime architecture

```text
Jupyter controller
        │ authenticated socket / PBS
        ▼
24 Python MPI workers (one process per rank)
        ├── read YAML and atm_in
        ├── build cubed-sphere grid and rank-local partition
        ├── allocate every persistent NumPy array
        ├── read vertical coordinates and generate DCMIP2016 state
        ├── initialize clock, constituents, and parameters
        ├── communicate through mpi4py
        └── Python StatePool: T/U/V/Q/PS, tendencies, grid, process state
                          │ CCPP standard_name / zero-copy arrays
                          ▼
                  Python DeviceRegistry
                          │ generated bind(C) adapter
                          ▼
             unchanged original Fortran scheme .so
```

Rank 0 reads fixed NetCDF inputs and broadcasts them with `mpi4py`; no CAM initialization or driver routine is called. Device manifests are discovered and shared libraries are loaded before initialization, but neither the ABI version functions nor numerical entrypoints are called until an explicit computational phase. Main dycore/support kernels remain in the separate model-kernel library.

## 1. Fixed v1 configuration

The model is intentionally fixed to CAM-SIMA `f8daa568eae2696b7c4ebff7768f02f5d097d9df`, FKESSLER, ne3np4.pg3, L30, 24 MPI ranks, one thread per rank, a 1800 s timestep, and the DCMIP2016 moist baroclinic wave.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os
import shutil

import freecam
from freecam import ModelConfig, NotebookSession
from freecam.model.validation import (
    compare_history_directories,
    compare_history_files,
)

repo = Path('/glade/work/ruitong/freeCAM')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
config = ModelConfig.from_yaml(config_path)
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)
oracle_path = os.environ.get('PYCAM_SIMA_ORACLE_DIR')
oracle_run = Path(oracle_path).expanduser().resolve() if oracle_path else None

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir = scratch / 'freecam/notebook_trials' / f'model-{stamp}' / 'run'
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, run_dir / 'atm_in')
history_dir = run_dir / 'history'

print('freecam', freecam.__version__)
print('run directory:', run_dir)
config.as_dict()

## 2. Start 24 Python MPI workers

On a Derecho login-node kernel, `start()` submits a PBS worker; inside an allocation it launches locally. It returns in state `INITIALIZED` before any model/kernel function or ABI version function has been called.

In [ ]:
if 'model' in globals() and model.running:
    model.close()

model = NotebookSession(
    config_path,
    run_dir=run_dir,
    history_dir=history_dir,
    python_executable=repo / '.venv/bin/python',
    log_path=run_dir / 'mpi-worker.log',
)
model.start()
assert model.initialized_native_calls == 0
assert model.initialized_abi_checked is False
print({
    'mode': model.launch_mode_used,
    'job': model.job_id,
    'ranks': model.ranks,
    'fields_and_aliases': len(model.field_names),
    'phase_status': model.phase_status,
    'scheme_status': model.scheme_status,
})

## 3. Inspect Python-owned fields

`model.fields[name]` returns a typed field handle. `get()` transfers a rank-local copy to the Notebook; `set()` writes an edited array back; `stats()` computes a compact summary in the MPI worker. `info()` reports owner, intent, dimensions, units, lifetime, category, and aliases.

In [ ]:
temperature_field = model.fields['air_temperature']
print(temperature_field.info())
print(temperature_field.stats(rank=0))
temperature_rank0 = temperature_field.get(rank=0)
temperature_rank0

In [ ]:
key_field_names = (
    'air_temperature',
    'eastward_wind',
    'northward_wind',
    'surface_pressure',
)
key_fields = {
    name: model.fields[name].info()
    for name in key_field_names
}
assert all(
    model.fields[name].info()['owner'] == 'python'
    for name in model.field_names
)
print('all fields and zero-copy aliases:', len(model.field_names))
key_fields

## 4. Explicit nstep=0 preparation

`prepare_initial_step()` executes dynamics-to-physics mapping, physics timestep initialization, the FKESSLER before-coupler schemes, and writes nstep=0 history. Calling `model.step()` directly from `INITIALIZED` performs this preparation automatically.

In [ ]:
status = model.prepare_initial_step()
print(status)
print('history files:', len(list(history_dir.glob('*.nc'))))

## 5. Inspect and run individual CCPP schemes

The Kessler CCPP suite is no longer a single opaque before/after block. All 19 `physics_before_coupler` schemes and all 5 `physics_after_coupler` schemes are separate Python interfaces. The `kessler` interface resolves fields by CCPP `standard_name`, passes their existing NumPy buffers through a generated adapter, and calls the original `external/CAM-SIMA/.../kessler.F90`; no copied Kessler equations are maintained in freeCAM. Every call is collective across all 24 workers, and every return checks that Python array addresses are unchanged.

In [ ]:
{
    'model_phases': model.phases.names,
    'before_coupler': model.physics.describe('before'),
    'after_coupler': model.physics.describe('after'),
}

In [ ]:
# Example for a fresh scheme-by-scheme session:
model.phases['dynamics_to_physics'].run()
model.phases['physics_timestep_initial'].run()
model.physics.scheme('calc_exner', group='before').run()
exner_after_scheme = model.fields['exner_function'].get(rank=0)
print('exner_function after calc_exner scheme:', exner_after_scheme)
model.physics.scheme('kessler', group='before').run()
state_after_kessler = model.fields[
    'physics_air_temperature'
].get(rank=0)

# These explicit methods are the opt-in to a non-validated sequence change.
model.physics['kessler_diagnostics'].disable()
model.physics['kessler'].move(after='kessler_update')
model.physics['kessler'].move(to_group='after')
model.physics.reset()  # restore the BFB-validated XML order

model.physics.sequence_safe

## 6. Optional field modification

Prognostic, tendency, and process fields may be changed at a Python boundary without replacing their NumPy storage. Static grid/topology fields reject writes unless `unsafe=True` is explicit. Any numerical edit intentionally breaks BFB.

In [ ]:
changed = temperature_field.get(rank=0)
changed[0, 0, 0, 0, 0] += 1.0e-6
temperature_field.set(changed, rank=0)
print(temperature_field.stats(rank=0))

#Static experiment, deliberately unsafe:
gll = model.fields['gll_node'].get(rank=0)
model.fields['gll_node'].set(gll, rank=0, unsafe=True)

## 7. Advance one complete 1800 s timestep

In [ ]:
step = model.step()
print('completed step:', step)
print(temperature_field.stats(rank=0))
print('history files:', len(list(history_dir.glob('*.nc'))))

## 8. Dynamically add a variable or physics device

A new variable is allocated collectively on all 24 ranks without moving any existing NumPy array. The Pythonic `model.fields` and `model.physics` façades translate friendly dimensions and placement options into the same strictly validated runtime protocol. A source `device.yaml` or prebuilt `device.json + .so` can be installed at a phase/scheme boundary.

In [ ]:
dynamic_name = 'notebook_diagnostic'
if dynamic_name not in model.field_names:
    model.fields.create(
        dynamic_name,
        dims=('column', 'level'),
        units='K',
        initial=0.0,
    )

dynamic_state = model.fields[dynamic_name].stats(rank=0)

# Set True to compile the included original-Fortran example collectively.
run_runtime_plugin = False
plugin_state = None
if run_runtime_plugin:
    plugin_name = 'runtime_temperature_offset'
    if plugin_name not in {item['name'] for item in model.physics_plugins}:
        model.physics.install(
            repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
            project_root=repo,
            after='kessler',
            inputs={
                'runtime_plugin_temperature': 240.0,
                'runtime_plugin_temperature_increment': 1.5,
            },
        )
    model.physics.scheme(plugin_name, group='before').run()
    plugin_state = model.fields[
        'ccpp_runtime_plugin_temperature'
    ].stats(rank=0)

{'dynamic_variable': dynamic_state, 'plugin': plugin_state}

## 9. Optional complete 50-step run

Run this only with untouched fields. `step(count)` remains one socket request while all 24 workers execute the same fixed plan.

In [ ]:
# if model.current_step < config.stop_n:
#     model.step(config.stop_n - model.current_step)
# print(model.current_step, len(list(history_dir.glob('*.nc'))))

## 10. Finalize and optionally compare history

The model run is self-contained. To compare against output from the pinned external CAM-SIMA executable, set `PYCAM_SIMA_ORACLE_DIR` to its history directory before running the setup cell.

In [ ]:
model.close()
print('closed:', run_dir)

In [ ]:
candidate_files = sorted(history_dir.glob('*.nc'))
if oracle_run is None:
    print('External comparison skipped; set PYCAM_SIMA_ORACLE_DIR to enable it')
else:
    for candidate in candidate_files:
        compare_history_files(oracle_run / candidate.name, candidate)
    print(f'BFB for all {len(candidate_files)} available model timestamps')

    if len(candidate_files) == 51:
        compare_history_directories(
            oracle_run, history_dir,
            expected_files=51, expected_numeric_variables=26,
        )
        print('FULL BFB: 50 steps, 51 timestamps, 26 numeric variables')

## 10. Recorded full validation

The uninterrupted model gate is stored in `validation/fkessler_model_bfb.json`; the original per-segment PBS fan-out and 25+25 restart gate are stored in `validation/dask_checkpoint_fanout.json`; and the direct-MPI single-allocation Dask gate is stored in `validation/dask_single_allocation.json`. Run the separate `try_dask_fanout.ipynb` Notebook for Dask experiments.

In [ ]:
evidence_paths = (
    repo / 'validation/fkessler_model_bfb.json',
    repo / 'validation/dask_checkpoint_fanout.json',
    repo / 'validation/dask_single_allocation.json',
)
{
    path.name: json.loads(path.read_text())
    for path in evidence_paths if path.exists()
}